# TPI 1: Adquisición y Análisis Lingüístico de Medios

**Modalidad:** Trabajo Práctico Individual (calificación numérica de 0 a 10).

**Fecha de entrega y exposición:** Jueves 16 de abril. Se realizará de manera expositiva en remoto frente a todo el grupo de estudiantes (aproximadamente 10 minutos por presentación) para que entre quienes participan veamos posibles soluciones.

**Duración estimada de codificación:** 2 horas

**Desafío general:**
Vas a construir un sistema en Python que adquiera textos de la web y transcriba audio, los analice lingüísticamente con spaCy, genere visualizaciones profesionales y exponga los resultados en un dashboard interactivo con Gradio. Este Trabajo Práctico Integrador fusiona las competencias de adquisición de datos y procesamiento de lenguaje natural.

**Dinámica de resolución: pair programming con IA**
La unidad de trabajo está formada por vos y un asistente de IA. La IA puede proponer estrategias, explicar errores, sugerir variantes y auditar código. No reemplaza tu pensamiento ni tu criterio. Toda decisión final, toda justificación y toda versión entregada tienen que estar bajo tu responsabilidad.

---

### AI Reflection Log — Plantilla obligatoria
Completá al menos una entrada en este registro por cada parte del laboratorio.

| Parte | Objetivo de la consulta | Prompt o pedido a la IA | Qué respondió (resumen) | Qué conservaste y por qué | Qué descartaste y por qué | Qué aprendiste |
|---|---|---|---|---|---|---|
| **Parte 1** | Estrategia para unificar fuentes de distinto formato en un DataFrame | Pedí estrategias para unificar texto de Trafilatura y Whisper | Propuso columnas mínimas comunes vs columnas separadas por tipo | Columnas mínimas: `titulo_o_fuente`, `texto`, `origen` — suficiente para el análisis | Columnas separadas por tipo — agrega complejidad innecesaria | Estandarizar el formato de entrada simplifica todo el pipeline posterior |
| **Parte 2** | Criterios para distinguir PER, ORG y LOC en spaCy | Pedí criterios explícitos para clasificar entidades por `ent.label_` | Explicó que PER=personas, ORG=organizaciones, LOC=lugares, MISC=otros | La estructura if/elif/else para clasificar entidades | Aceptar los resultados del modelo sin revisión crítica | El modelo comete errores: clasifica empresas como personas y lugares como organizaciones |
| **Parte 3** | Elegir entre WordCloud y Barplot para frecuencias | Pedí criterios aplicando Data-Ink Ratio | Explicó que Barplot permite comparación precisa, WordCloud es visual pero impreciso | Lollipop chart — combina precisión y claridad visual | WordCloud como visualización principal de frecuencias | Data-Ink Ratio favorece gráficos que permiten leer valores exactos |
| **Parte 4** | Separar exportación CSV y JSON | Pedí justificación para dos formatos de exportación | Explicó que CSV es tabular/plano y JSON es jerárquico/estructurado | Drop de columna `doc` antes de exportar CSV — no es serializable | Exportar todo en un solo formato | La columna `doc` de spaCy no se puede serializar directamente |
| **Parte 5** | Layout del dashboard Gradio | Pedí tres opciones de estructura | Propuso Pestañas, Columna vertical y Acordeón | Pestañas — separa métricas de exploración sin sobrecargar la vista | Columna vertical — todo junto es difícil de leer; Acordeón — oculta información relevante | Las pestañas permiten separar contextos de uso sin perder acceso a los datos |


In [1]:
# PASO 0: Instalación de las librerías necesarias
# Ejecutá esta celda una sola vez.
!pip install spacy trafilatura pandas matplotlib seaborn plotly wordcloud openai-whisper yt-dlp gradio -q
!python -m spacy download es_core_news_lg -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')


In [2]:
import spacy
import pandas as pd
import trafilatura
import whisper
import json
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from collections import Counter
from wordcloud import WordCloud
import os

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## Parte 1: Adquisición Multimodal del Corpus

**Objetivo:** Construir funciones que permitan alimentar el pipeline obteniendo datos desde tres vías: scraping en vivo (Trafilatura), transcripción de audio (Whisper), y carga de JSON previo. Luego, unificarlas en un único DataFrame.

> [!IMPORTANT]  
> **Dilema de diseño (Restricción generativa)**
> Antes de escribir el código de unificación, consultá a tu asistente de IA. Pedile estrategias para lidiar con las diferencias de formato al unificar un texto transcrito de un podcast (audio) con una nota periodística scrapeada (Trafilatura) en un solo DataFrame. 
> Elegí un enfoque para alinear las columnas, justificalo a continuación y registrá la consulta en tu *AI Reflection Log*.

**Escribí tu justificación acá:**

**Justificación del enfoque de unificación:**

Las fuentes tienen estructuras distintas: Trafilatura produce texto periodístico con puntuación y párrafos definidos, mientras que Whisper genera transcripciones continuas con marcas de oralidad y sin estructura clara.

Elegí la **Estrategia de columnas mínimas comunes**: unificar en un DataFrame con tres columnas — `titulo_o_fuente`, `texto` y `origen` — que todas las fuentes pueden proveer sin importar su formato. Las diferencias internas de cada texto quedan dentro de la columna `texto` y se gestionan en el preprocesamiento lingüístico posterior con spaCy.

Descarté una estructura con columnas separadas por tipo porque agrega complejidad innecesaria para el análisis que requiere este TPI.


In [3]:
# 1.1 Scraping en vivo
def extraer_noticias_web(urls):
    """Extrae el texto de una lista de URLs usando Trafilatura"""
    noticias = []
    # PASO 1: Iterá sobre cada URL, descargá y extraé el contenido con trafilatura.
    # Recordá incluir manejo de excepciones (try/except) para no detener la ejecución si una URL falla.
    for url in urls:
        try:
            contenido = trafilatura.fetch_url(url)
            if contenido:
                texto = trafilatura.extract(contenido, include_comments=False, include_tables=False)
                noticias.append({'titulo_o_fuente': url, 'texto': texto, 'origen': 'web'})
            else:
                print(f"No se pudo descargar el contenido de {url}")
        except Exception as e:
            print(f"Error al procesar {url}: {e}")
    return noticias

In [4]:
# 1.2 Transcripción de Audio
def transcribir_audio_youtube(url_video):
    """Descarga el audio de un video de YouTube y lo transcribe usando Whisper"""
    import subprocess
    import yt_dlp
    
    # PASO 2: Utilizá yt-dlp para descargar el audio.
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'audio_descargado.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True
        }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url_video])

    # Luego, cargá el modelo 'small' o 'base' de whisper y transcribí el archivo generado.
    modelo = whisper.load_model("small")
    resultado = modelo.transcribe("audio_descargado.mp3", language='es')
    os.remove("audio_descargado.mp3")  # Limpieza del archivo temporal
    texto =resultado['text']
    return [{'titulo_o_fuente': url_video, 'texto': texto, 'origen': 'audio'}]
 

In [5]:
# 1.3 Carga de JSON local
def cargar_json_previo(ruta_json):
    """Carga un corpus pre-extraído en formato JSON"""
    # PASO 3: Utilizá pandas (pd.read_json) o la librería json nativa para cargar los datos.
    noticias = []
    with open(ruta_json, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    for item in datos:
            noticias.append({
            'titulo_o_fuente': item.get('titulo', ''),
            'texto': item.get('titulo', ''),
            'origen': 'json'
        })
    return noticias

In [6]:
# 1.4 Consolidación
def unificar_corpus(datos_web, datos_audio, datos_json):
    """Unifica las tres fuentes en un DataFrame con columnas estándar"""
    # PASO 4: Implementá la estrategia que decidiste en el dilema de diseño.
    # El DataFrame final debería tener al menos: 'titulo_o_fuente', 'texto', 'origen' ('web', 'audio', 'json')
    todos = datos_web + datos_audio + datos_json
    df_unificado = pd.DataFrame(todos)
    return df_unificado

# ---- Espacio para pruebas ----
# Probá tus funciones acá con al menos 1 url web y 1 video corto.
# Prueba con una URL real
datos_web = extraer_noticias_web([
    "https://www.lanacion.com.ar/politica/uno-por-uno-los-bienes-que-la-justicia-le-decomisara-a-cristina-kirchner-nid24042026/"
])

datos_json = cargar_json_previo("../002_adquisicion_corpus/lanacion_portada.json")

df_corpus = unificar_corpus(datos_web, [], datos_json)
print(df_corpus.shape)
df_corpus.head()


(144, 3)


,titulo_o_fuente,texto,origen
0,https://www.lanacion.com.ar/politica/uno-por-u...,"Uno por uno, los bienes que la Justicia le dec...",web
1,Análisis. Un ladrillo flojo sacude la arquitec...,Análisis. Un ladrillo flojo sacude la arquitec...,json
2,Tensión con la prensa. El Gobierno prohíbe el ...,Tensión con la prensa. El Gobierno prohíbe el ...,json
3,Viaje familiar. La Justicia confirmó que Adorn...,Viaje familiar. La Justicia confirmó que Adorn...,json
4,"Máxima tensión. Con un video, Irán exhibe su c...","Máxima tensión. Con un video, Irán exhibe su c...",json


> **Pausa de auditoría:**
> Revisá tu DataFrame consolidado (`df_corpus.head()`). ¿Cómo afectó la falta de puntuación o marcas de oralidad en la transcripción de Whisper respecto del texto estructurado de las noticias? Revisá las columnas generadas. ¿Perdiste información contextual al unificarlas?

Respuesta

El DataFrame consolidado muestra las tres columnas correctamente (`titulo_o_fuente`, `texto`, `origen`). 

Sin la fuente de audio todavía incorporada, la comparación es parcial. Sin embargo, anticipamos que el texto de Whisper tendrá menos puntuación y más marcas de oralidad ("eh", "bueno", "o sea") que el texto periodístico de Trafilatura, lo que puede afectar la segmentación de oraciones en spaCy (`doc.sents`). No se perdió información contextual al unificar porque la columna `origen` permite identificar la fuente de cada registro.


## Parte 2: Análisis Lingüístico con spaCy

**Objetivo:** Encapsular el análisis en una clase reutilizable, distinguiendo qué atributos del modelo de spaCy sirven para resolver cada necesidad.

> [!IMPORTANT]
> **Dilema de diseño**
> Pedile a la IA que te proponga criterios explícitos para distinguir entre entidades de tipo 'PER', 'ORG' y 'LOC' a partir de la propiedad `ent.label_` de spaCy. Después verificá si el modelo realmente las clasifica así en la práctica.
> Anotá en el log si encontraste diferencias entre la teoría que te dio la IA y la salida real del modelo.

In [7]:
class AnalizadorCorpus:
    def __init__(self, df, modelo_spacy="es_core_news_lg"):
        self.df = df
        print("Cargando modelo de lenguaje...")
        self.nlp = spacy.load(modelo_spacy)
        
        # Procesamos la columna 'texto' al instanciar la clase
        print("Procesando los textos con spaCy...")
        # PASO 1: Creá una nueva columna en el DataFrame llamada 'doc' que contenga el objeto procesado por self.nlp()
        self.df['doc'] = list(self.nlp.pipe(self.df['texto']))


    def extraer_entidades(self):
        """Devuelve las entidades agrupadas por tipo, contabilizando total de apariciones"""
        # PASO 2: Recorré los 'doc' del DataFrame y armá un diccionario o lista con las entidades halladas.
        entidades = {
            'PERSONAS': [],
            'ORGANIZACIONES': [],
            'LUGARES': [],
            'OTROS': []
        }
        for doc in self.df['doc']:  
            for ent in doc.ents:
                if ent.label_ == 'PER':
                    entidades['PERSONAS'].append(ent.text)
                elif ent.label_ == 'ORG':
                    entidades['ORGANIZACIONES'].append(ent.text)
                elif ent.label_ == 'LOC':
                    entidades['LUGARES'].append(ent.text)   
                else:
                    entidades['OTROS'].append(ent.text)
        return entidades



    def extraer_verbos_principales(self, n=15):
        """Devuelve los 'n' verbos lematizados más frecuentes de todo el corpus"""
        # PASO 3: Filtrá tokens que sean VERB y no sean stopwords, extraé su lema y contalos.
        verbos_principales=[]
        for doc in self.df['doc']:
            for token in doc:
                if token.pos_ == 'VERB' and not token.is_stop:
                    verbos_principales.append(token.lemma_)
        conteo_verbos = Counter(verbos_principales)
        return conteo_verbos.most_common(n)

    def extraer_palabras_clave(self, n=20):
        """Devuelve sustantivos y nombres propios lematizados y filtrados"""
        # PASO 4: Implementá una lógica superior a la del Lab 009 (donde usamos stopwords crudas).
        # Filtrá por categorías gramaticales relevantes (NOUN, PROPN, ADJ) omitiendo puntuación y stopwords.
            
        palabras_clave = []
        for doc in self.df['doc']:
            for token in doc:
                if (token.pos_ in ['NOUN', 'PROPN', 'ADJ'] 
                    and token.is_alpha 
                    and not token.is_stop 
                    and len(token.text) > 2):
                    palabras_clave.append(token.lemma_.lower())
        conteo = Counter(palabras_clave)
        return conteo.most_common(n)

        
    def estadisticas_corpus(self):
        """Genera un diccionario con métricas generales del corpus"""
        # PASO 5: Calculá total de tokens, tamaño del vocabulario único (lemas) y cantidad de oraciones.
        
        total_tokens = sum(len(doc) for doc in self.df['doc'])
        total_oraciones = sum(len(list(doc.sents)) for doc in self.df['doc'])
        palabras_unicas = len(set(
            token.lemma_.lower() 
            for doc in self.df['doc'] 
            for token in doc 
            if token.is_alpha and not token.is_stop
        ))
        return {
            'total_tokens': total_tokens,
            'total_oraciones': total_oraciones,
            'palabras_unicas': palabras_unicas,
            'longitud_promedio_oracion': total_tokens / total_oraciones if total_oraciones > 0 else 0
        }


# ---- Espacio para pruebas ----
# analizador = AnalizadorCorpus(df_corpus)
# print(analizador.estadisticas_corpus())

> **Pausa de auditoría:**
> Compará el desempeño de spaCy sobre una noticia escrita versus sobre el texto transcrito con Whisper. ¿Dónde cometió más fallas el modelo algorítmico al intentar agrupar oraciones (sents) o detectar nombres propios? ¿Por qué creés que se da este fenómeno?

## Parte 3: Visualización Profesional

**Objetivo:** Aplicar principios de Data-Ink Ratio, accesibilidad y jerarquía visual para comunicar hallazgos efectivamente, en lugar de imprimir datos planos.

> [!IMPORTANT]
> **Dilema de diseño**
> Consultá a la IA: ¿conviene usar un *WordCloud* o un *Barplot* para mostrar frecuencias de palabras clave en un informe dirigido a toma de decisiones? Justificá tu elección aplicando el principio de Data-Ink Ratio.

**Escribí tu justificación acá:**

Elegí el **Barplot/Lollipop** sobre el WordCloud aplicando el principio de Data-Ink Ratio. El WordCloud es visualmente atractivo pero impreciso — no permite comparar valores exactos entre palabras. El Lollipop chart muestra las mismas frecuencias con mayor claridad y permite al lector leer valores concretos, lo que es más adecuado para un informe de toma de decisiones.


In [8]:
# Configuración base de accesibilidad visual
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.1)
COLOR_ACENTO = sns.color_palette("colorblind")[0]
COLOR_BASE = '#b0b0b0'

def visualizar_origen(df):
    """Genera un barplot con el origen de los datos o las secciones"""
    # PASO 1: Generá un barplot horizontal orientado a objetos (fig, ax) usando Seaborn.
    # Aplicá el COLOR_ACENTO a la barra principal (la de mayor count).
    # Despintá los bordes molestos con sns.despine()
    
    fig, ax = plt.subplots(figsize=(8, 4))
    conteo = df['origen'].value_counts().reset_index()
    conteo.columns = ['origen', 'cantidad']
    colores = [COLOR_ACENTO if i == 0 else COLOR_BASE for i in range(len(conteo))]
    sns.barplot(data=conteo, x='cantidad', y='origen', palette=colores, ax=ax)
    ax.set_title('Distribución de fuentes del corpus', fontsize=12)
    ax.set_xlabel('Cantidad de textos')
    ax.set_ylabel('')
    sns.despine()
    plt.tight_layout()
    #plt.show()
    return fig


def visualizar_palabras_clave_lollipop(palabras_clave):
    """Genera un Lollipop Chart de las palabras clave lematizadas"""
    # PASO 2: Construí el gráfico estructurado (Lollipop) para las palabras clave extraídas en la Parte 2.
    # Recordá que el lollipop se arma combinando ax.hlines y ax.plot.
    
    palabras = [p for p, _ in reversed(palabras_clave)]
    frecuencias = [f for _, f in reversed(palabras_clave)]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hlines(y=range(len(palabras)), xmin=0, xmax=frecuencias, color=COLOR_ACENTO, linewidth=1.5)
    ax.plot(frecuencias, range(len(palabras)), 'o', color=COLOR_ACENTO, markersize=7)
    ax.set_yticks(range(len(palabras)))
    ax.set_yticklabels(palabras)
    ax.set_title('Palabras clave del corpus', fontsize=12)
    ax.set_xlabel('Frecuencia')
    sns.despine()
    plt.tight_layout()
    #plt.show()
    return fig


def visualizar_entidades_plotly(entidades_dict):
    """Genera un panel interactivo con Plotly para las entidades más comunes"""
    # PASO 3: Resolvelo utilizando go.Bar y devolvé el objeto figura (fig) de Plotly
    # para usarlo posteriormente en Gradio.
    
    todas = entidades_dict['PERSONAS'] + entidades_dict['ORGANIZACIONES'] + entidades_dict['LUGARES']
    conteo = Counter(todas).most_common(15)
    nombres = [e for e, _ in conteo]
    frecuencias = [f for _, f in conteo]
    fig = go.Figure(go.Bar(x=frecuencias, y=nombres, orientation='h'))
    fig.update_layout(title='Entidades más comunes', xaxis_title='Frecuencia', yaxis=dict(autorange='reversed'))
    return fig


> **Pausa de auditoría:**
> Revisá tu visualización. ¿Es accesible? El uso de la paleta 'colorblind' asegura que ciertos grados de daltonismo no impidan la lectura cromática, pero ¿el tamaño de fuente y la proporción de la figura se leen correctamente sin forzar la vista? ¿Qué cambiarías si tuvieras que publicarlo en un artículo científico?

## Parte 4: Pipeline Integrado (Orquestación)

**Objetivo:** Orquestar los componentes desarrollados en un flujo lógico unificado y persistir los hallazgos en formato estructurado. Todo sistema analítico debe poder guardar su estado final de forma estática.

In [9]:
class PipelineMediatico:
    def __init__(self, urls_web=None, url_audio=None, ruta_json=None):
        self.urls_web = urls_web or []
        self.url_audio = url_audio
        self.ruta_json = ruta_json
        self.df = None
        self.analizador = None
        
    def ejecutar_pipeline(self):
        """Orquesta la adquisición, unificación y análisis"""
        # PASO 1: Orquestá las llamadas a las funciones de la Parte 1.
        datos_web = extraer_noticias_web(self.urls_web)
        datos_audio = transcribir_audio_youtube(self.url_audio) if self.url_audio else []
        datos_json = cargar_json_previo(self.ruta_json) if self.ruta_json else []
        self.df = unificar_corpus(datos_web, datos_audio, datos_json)
       
        # PASO 2: Instanciá AnalizadorCorpus y derivale el DataFrame resultante para procesar.
        self.analizador = AnalizadorCorpus(self.df)
        print("Pipeline ejecutado exitosamente.")
        
        
    def generar_reporte_y_exportar(self, ruta_csv="corpus_resultante.csv", ruta_json="estadisticas.json"):
        """Exporta el dataframe y un JSON analítico"""
        # PASO 3: Persistí self.df como CSV.
        # ¡OJO! La columna 'doc' de spaCy no es serializable, deberías dropearla o extraer sus textos antes de guardar.
        df_exportar = self.df.drop(columns=['doc'], errors='ignore')
        df_exportar.to_csv(ruta_csv, index=False)

        # PASO 4: Persistí las estadísticas y el diccionario de entidades devueltas por el Analizador como JSON local.
        estadisticas = self.analizador.estadisticas_corpus()
        entidades = self.analizador.extraer_entidades()
        with open(ruta_json, 'w', encoding='utf-8') as f:
            json.dump({'estadisticas': estadisticas, 'entidades': entidades}, f, ensure_ascii=False, indent=2)

        print(f"Exportado: {ruta_csv} y {ruta_json}")

# ---- Espacio para pruebas ----


pipeline = PipelineMediatico(
    urls_web=[
        "https://www.lanacion.com.ar/politica/uno-por-uno-los-bienes-que-la-justicia-le-decomisara-a-cristina-kirchner-nid24042026/"
    ],
     url_audio="https://www.youtube.com/watch?v=UMz_DNlpy30",
    ruta_json="../002_adquisicion_corpus/lanacion_portada.json"
)

pipeline.ejecutar_pipeline()
analizador = pipeline.analizador
print(analizador.estadisticas_corpus())


c:\Users\54116\Documents\IFTS24_Lab_PLN\ifts24-lab-pln-2026\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Cargando modelo de lenguaje...
Procesando los textos con spaCy...
Pipeline ejecutado exitosamente.
{'total_tokens': 4592, 'total_oraciones': 361, 'palabras_unicas': 1230, 'longitud_promedio_oracion': 12.720221606648199}


> **Pausa de auditoría:**
> Imaginá que un equipo de periodismo de datos de tu facultad te pide el corpus procesado. ¿Qué información necesitaban ellos en el CSV plano versus qué preferiste consolidar en el JSON jerárquico? Pensá por qué separamos esas dos naturalezas de exportación y registralo.

## Parte 5: Dashboard Interactivo con Gradio

**Objetivo:** Diseñar una interfaz interactiva que reaccione dinámicamente frente al usuario, conectando el backend analítico con componentes preconstruidos de frontend.

> [!IMPORTANT]
> **Dilema de diseño**
> ¿Qué componentes elegirías para cada salida? Pedile a la IA tres layouts de estructura (por ejemplo: Pestañas vs. Columna vertical vs. Acordeón) para tu dashboard. Elegí el que consideres mejor para la experiencia de lectura evaluativa y descartá explícitamente los otros dos argumentando tu postura técnica.

**Escribí tu justificación acá:**

Elegí la estructura de **Pestañas** sobre Columna vertical y Acordeón. La columna vertical acumula todo en una sola vista y dificulta la lectura. El Acordeón oculta información que puede ser relevante. Las pestañas separan claramente dos contextos de uso: métricas generales del corpus y exploración de entidades, permitiendo al usuario enfocarse en una tarea a la vez sin perder acceso a la otra.


In [ ]:
# PASO 1: Diseñá el bloque principal de gr.Blocks() interactuando con los métodos de la clase AnalizadorCorpus.
# Sugerencia: Utilizá pestañas (gr.Tab) para separar "Métricas Generales" de "Filtros e Interacción".

with gr.Blocks(theme=gr.themes.Soft()) as dashboard_medios:
    gr.Markdown("# Explorador de Agenda Mediática")
    
    with gr.Tab("Panorama y Métricas"):
        # Incluí acá la visualización de frecuencias y orígenes, acompañando un gr.DataFrame con métricas generales.
        gr.Markdown("## Métricas generales del corpus")
        btn_metricas = gr.Button("Generar métricas")
        tabla_metricas = gr.Dataframe(label="Estadísticas")
        grafico_origen = gr.Plot(label="Distribución de fuentes")
        grafico_palabras = gr.Plot(label="Palabras clave")
        
        def mostrar_metricas():
            stats = analizador.estadisticas_corpus()
            df_stats = pd.DataFrame([stats])
            fig_origen = visualizar_origen(analizador.df)
            palabras = analizador.extraer_palabras_clave()
            fig_palabras = visualizar_palabras_clave_lollipop(palabras)
            return df_stats, fig_origen, fig_palabras
        
        btn_metricas.click(mostrar_metricas, outputs=[tabla_metricas, grafico_origen, grafico_palabras])
    
        
    with gr.Tab("Explorador de Entidades"):
        # Desarrollá un textbox para ingresar una entidad y un botón que dispare
        # un filtrado, mostrando sólo las oraciones dentro de los textos donde se mencionó dicha entidad.
        gr.Markdown("## Buscá entidades en el corpus")
        entidad_input = gr.Textbox(label="Nombre de entidad")
        btn_buscar = gr.Button("Buscar")
        resultado_entidades = gr.Textbox(label="Oraciones donde aparece", lines=10)
        grafico_entidades = gr.Plot(label="Entidades más comunes")
        
        def buscar_entidad(nombre):
            oraciones = []
            for doc in analizador.df['doc']:
                for sent in doc.sents:
                    if nombre.lower() in sent.text.lower():
                        oraciones.append(sent.text.strip())
            entidades = analizador.extraer_entidades()
            fig = visualizar_entidades_plotly(entidades)
            return "\n\n".join(oraciones[:5]) if oraciones else "No se encontraron menciones.", fig
        
        btn_buscar.click(buscar_entidad, inputs=entidad_input, outputs=[resultado_entidades, grafico_entidades])

# Descomentá la siguiente línea cuando el bloque esté terminado
dashboard_medios.launch()

C:\Users\54116\AppData\Local\Temp\ipykernel_23828\1232043184.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as dashboard_medios:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


C:\Users\54116\AppData\Local\Temp\ipykernel_23828\2721338719.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=conteo, x='cantidad', y='origen', palette=colores, ax=ax)
C:\Users\54116\AppData\Local\Temp\ipykernel_23828\2721338719.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=conteo, x='cantidad', y='origen', palette=colores, ax=ax)


---
## Cierre Formal y Checklist de Entrega

1. ¿Corriste el pipeline de principio a fin, comprobando que las funciones se anidan y comparten el DataFrame correctamente?
2. ¿Tu *AI Reflection Log* evidencia que cuestionaste a la IA y al modelo algorítmico, o todas tus celdas dicen "me devolvió un código y lo usé"?
3. ¿Revisaste el impacto visual de los gráficos garantizando que minimizan la "tinta algorítmica" (Data-Ink Ratio)?
4. ¿Justificaste tus decisiones de arquitectura técnica en el código de orquestación y exportación?

Si respondiste positivamente: felicitaciones, completaste el **TPI 1** demostrando un uso constructivo de la IA, asumiendo un rol profesional capaz de dirigir la automatización de forma estratégica e informada.